# Experiment 3 adaptation: 10-feature income baseline and Missing-Feature MIA
Target: California ACS 2023, nominal personal income >$50,000. Auxiliary: California ACS 2014 only.
Keep ACSIncome eligibility filters. Inputs: AGEP, COW, SCHL, MAR, OCCP, POBP, WKHP, SEX, RAC1P, HISP.
The former RELP field is unavailable in 2023; HISP supplies the tenth common field.
100,000 target training records; validation/test/MIA-member/MIA-nonmember sets each 10,000.
16 shadows cycle RF (6), gradient boosting (5), XGBoost (5), retaining Experiment 3 settings.
Attack: the same four paper-inspired features and RF, grouped auxiliary record splits.
This user-authorized run tests the full-feature baseline and all 15 subsets of 1–4 withheld
fields from COW, WKHP, MAR and SCHL. No income label is hidden or used as an input.

## Explicit adaptation choices
Masked AE trained on an 80% auxiliary split only, with the other 20% held out.
Architecture: input→128→64→64→128→output; ReLU hidden layers, MSE on all encoded coordinates,
50 epochs, Adam lr=.001, batch size 256. Numerics MinMax-scaled, categories one-hot encoded.
Uniformly sample 1–4 whole original fields per row from all ten predictors each training iteration;
replace all coordinates for those fields with -1. This sampling rule and decoding are project choices.
Decode categorical blocks by argmax; clip numeric reconstructions to auxiliary train ranges and round.
Only fill withheld fields; preserve observed inputs. Candidate labels and membership never enter the AE.
Train the attack on complete auxiliary records and freeze its classifier and auxiliary-selected thresholds
for every missing-feature scenario (paper Sections 4.2–4.3). Do not tune using target membership results.
The blind walk starts at each completed record and uses the existing native-field perturbation protocol.
Report per-mask results and macro averages by missing-field count, plus imputation error.
This is an income/ACS adaptation; the paper used gender/GSS with a different feature schema.


In [1]:
from pathlib import Path
import itertools, json, time, joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.missing_feature_mia import (FEATURES, WITHHELD, LABEL, load_year, prepare,
    train_shadows, MaskedAutoencoder, run_mask_evaluations)
from src.data import split_acs_data, subset_target_training
from src.exp3_checkpoints import cached_target_train, save
from src.training import evaluate_model
from src.shadow_models import summarize_shadow_models
from src.paper_attack_features import (collect_shadow_paper_features,
    collect_target_paper_features, make_probability_oracle)
from src.membership_attack import train_membership_classifier, evaluate_attack
torch.set_num_threads(4)
SEED = 42
TRAIN_SIZE = 100_000
RUN_DIR = Path('artifacts/exp_3_missing_features_10inputs_2023_2014')
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CONFIG = dict(hidden_dims=(512,256,128),dropout=0.0,epochs=100,batch_size=256,
    learning_rate=1e-3,weight_decay=0.0,seed=SEED,device='cpu',early_stopping=False,
    restore_best=False,monitor_train=True)
WALK_CONFIG = dict(max_steps=20,repeats=2,categorical_change_probability=.05,batch_size=512,seed=SEED)
SPLIT_CONFIG = dict(validation_size=10000,test_size=10000,mia_nonmember_size=10000,
    mia_member_size=10000,seed=SEED)
print('Target predictors:', FEATURES)
assert len(FEATURES)==10 and LABEL not in FEATURES


Target predictors: ['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'WKHP', 'SEX', 'RAC1P', 'HISP']


## Load native-year data and create disjoint target partitions

In [2]:
source = load_year(2023)
auxiliary = load_year(2014)
assert set(source.index).isdisjoint(auxiliary.index)
partitions = subset_target_training(split_acs_data(source,**SPLIT_CONFIG),train_size=TRAIN_SIZE,seed=SEED)
assert set(partitions['mia_members'].index) <= set(partitions['train'].index)
for name in ['validation','test','mia_nonmembers']:
    assert set(partitions[name].index).isdisjoint(partitions['train'].index)
pp, arrays = prepare(partitions)
save(RUN_DIR/'partitions_preprocessor.joblib',dict(preprocessor=pp,
    partitions={name:frame.index.to_numpy() for name,frame in partitions.items()}))
print('Eligible 2023:',len(source),'| Eligible 2014:',len(auxiliary))
print('Encoded target inputs:',arrays['train']['X'].shape[1])
display(pd.DataFrame({name:dict(records=len(frame),positive_rate=frame[LABEL].mean())
    for name,frame in partitions.items()}).T)


Eligible 2023: 203278 | Eligible 2014: 183941
Encoded target inputs: 821


,records,positive_rate
validation,10000.0,0.49840
test,10000.0,0.49840
mia_nonmembers,10000.0,0.49840
train,100000.0,0.49845
mia_members,10000.0,0.49840


## Train the same target architecture for 100 epochs; retain final weights

In [3]:
target, target_history, target_metadata = cached_target_train(arrays,TRAIN_CONFIG,RUN_DIR)
target_metrics = {name:evaluate_model(target,arrays[name]['X'],arrays[name]['y'])
    for name in ['train','validation','test']}
display(pd.DataFrame(target_metrics).T)


Epoch 001: train loss=0.4270, validation loss=0.4115, validation accuracy=0.8085


Epoch 002: train loss=0.3941, validation loss=0.4084, validation accuracy=0.8137


Epoch 003: train loss=0.3780, validation loss=0.4089, validation accuracy=0.8142


Epoch 004: train loss=0.3587, validation loss=0.4172, validation accuracy=0.8076


Epoch 005: train loss=0.3334, validation loss=0.4367, validation accuracy=0.8057


Epoch 006: train loss=0.3037, validation loss=0.4727, validation accuracy=0.8019


... (91 outputs left out)

Epoch 098: train loss=0.0245, validation loss=2.1015, validation accuracy=0.7726


Epoch 099: train loss=0.0278, validation loss=2.1239, validation accuracy=0.7713


Epoch 100: train loss=0.0266, validation loss=2.0298, validation accuracy=0.7758
Completed target checkpoint saved.


,accuracy,roc_auc,log_loss
train,0.9894,0.999643,0.024344
validation,0.7758,0.840932,2.029802
test,0.7793,0.841954,2.004685


## Train 16 auxiliary shadows and verify the native-input query interface

In [4]:
shadows = train_shadows(auxiliary,RUN_DIR,count=16,seed=SEED)
for name,result in shadows.items():
    audit = result['attack_data'].iloc[:32]
    oracle = make_probability_oracle(result['model'],result['preprocessor'],schema='acs')
    np.testing.assert_allclose(oracle(auxiliary.loc[audit.record_id,FEATURES]),
        audit[['p_income_le_50k','p_income_gt_50k']].to_numpy(),rtol=1e-5,atol=1e-6)
summarize_shadow_models(shadows).to_csv(RUN_DIR/'shadow_metrics.csv')
display(summarize_shadow_models(shadows))


Training shadow 1/16: random_forest_01


Saved shadow: random_forest_01


Training shadow 2/16: gradient_boosting_02


Saved shadow: gradient_boosting_02


Training shadow 3/16: xgboost_03


Saved shadow: xgboost_03


... (23 outputs left out)

Saved shadow: xgboost_15


Training shadow 16/16: random_forest_16


Saved shadow: random_forest_16


records  accuracy   roc_auc  log_loss
shadow_model         partition                                        
random_forest_01     train         73576  0.852601  0.930695  0.355723
                     validation    36789  0.818451  0.894224  0.402215
                     nonmembers    73576  0.817291  0.893548  0.402855
gradient_boosting_02 train         73576  0.813907  0.891095  0.400339
                     validation    36789  0.814048  0.888499  0.404103
                     nonmembers    73576  0.810998  0.888329  0.404280
xgboost_03           train         73576  0.825364  0.901976  0.379667
                     validation    36789  0.821577  0.897758  0.385957
                     nonmembers    73576  0.818840  0.895319  0.389601
random_forest_04     train         73576  0.852506  0.930488  0.356577
                     validation    36789  0.818342  0.894683  0.401551
                     nonmembers    73576  0.817672  0.893552  0.402973
gradient_boosting_05 train         73576  0.813594  0.891328  0.399891
                     validation    36789  0.812716  0.889181  0.402830
                     nonmembers    73576  0.812085  0.888435  0.404147
xgboost_06           train         73576  0.826587  0.903837  0.376285
                     validation    36789  0.818614  0.894815  0.390081
                     nonmembers    73576  0.819534  0.896010  0.388188
random_forest_07     train         73576  0.851718  0.930343  0.356322
                     validation    36789  0.816086  0.893440  0.402680
                     nonmembers    73576  0.817468  0.893625  0.403634
gradient_boosting_08 train         73576  0.816734  0.893463  0.396803
                     validation    36789  0.809264  0.885884  0.407365
                     nonmembers    73576  0.811555  0.887069  0.405553
xgboost_09           train         73576  0.824508  0.902420  0.378880
                     validation    36789  0.820191  0.895327  0.389607
                     nonmembers    73576  0.820526  0.897091  0.386914
random_forest_10     train         73576  0.851174  0.929860  0.356518
                     validation    36789  0.818152  0.893290  0.403737
                     nonmembers    73576  0.818772  0.895722  0.400407
gradient_boosting_11 train         73576  0.814355  0.890594  0.401442
                     validation    36789  0.811058  0.887595  0.405391
                     nonmembers    73576  0.812887  0.888522  0.403958
xgboost_12           train         73576  0.827158  0.903697  0.376669
                     validation    36789  0.820544  0.896724  0.387333
                     nonmembers    73576  0.818392  0.895513  0.388905
random_forest_13     train         73576  0.850590  0.929592  0.357441
                     validation    36789  0.817337  0.895746  0.400348
                     nonmembers    73576  0.818963  0.894419  0.401927
gradient_boosting_14 train         73576  0.814627  0.890721  0.400904
                     validation    36789  0.811465  0.888004  0.404515
                     nonmembers    73576  0.812031  0.888295  0.404445
xgboost_15           train         73576  0.825378  0.903571  0.377015
                     validation    36789  0.818071  0.895073  0.389700
                     nonmembers    73576  0.819602  0.895956  0.388122
random_forest_16     train         73576  0.854450  0.931098  0.355727
                     validation    36789  0.816684  0.892181  0.405213
                     nonmembers    73576  0.817835  0.893685  0.402883

## Full-feature membership attack: freeze classifier and auxiliary thresholds

In [5]:
shadow_features,shadow_diagnostics = collect_shadow_paper_features(
    shadows,auxiliary,per_class=2000,config=WALK_CONFIG,seed=SEED)
attack = train_membership_classifier(shadow_features,seed=SEED,algorithm='random_forest',feature_set='paper')
save(RUN_DIR/'attack.joblib',attack)
target_features,target_diagnostics = collect_target_paper_features(target,pp,partitions,config=WALK_CONFIG,seed=SEED)
baseline,baseline_scores = evaluate_attack(attack['classifier'],target_features,
    threshold=attack['accuracy_threshold'],low_fpr_threshold=attack['low_fpr_threshold'])
save(RUN_DIR/'baseline.joblib',dict(features=target_features,scores=baseline_scores,metrics=baseline))
display(pd.DataFrame({'full_feature_target_mia':baseline}).T)


Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (159 outputs left out)

Attack feature extraction: 19,456/20,000


Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
full_feature_target_mia,20000.0,10000.0,0.6049,0.6049,0.645108,0.0017,0.501622,2404.0,7596.0,306.0,9694.0,0.57812,0.2181,0.1341


## Train the auxiliary-only masked autoencoder

In [6]:
completer = MaskedAutoencoder().fit(auxiliary,seed=SEED,epochs=50)
assert set(completer.train_ids).isdisjoint(completer.validation_ids)
assert set(completer.train_ids).isdisjoint(source.index)
save(RUN_DIR/'masked_autoencoder.joblib',completer)
pd.DataFrame(completer.history).to_csv(RUN_DIR/'autoencoder_history.csv',index=False)
display(pd.DataFrame(completer.history).iloc[::10])


Autoencoder epoch 01/50: loss=0.005379


Autoencoder epoch 02/50: loss=0.003831


Autoencoder epoch 03/50: loss=0.003526


Autoencoder epoch 04/50: loss=0.003384


Autoencoder epoch 05/50: loss=0.003300


Autoencoder epoch 06/50: loss=0.003238


... (41 outputs left out)

Autoencoder epoch 48/50: loss=0.002582


Autoencoder epoch 49/50: loss=0.002574


Autoencoder epoch 50/50: loss=0.002577


,epoch,reconstruction_loss
0,1,0.005379
10,11,0.002997
20,21,0.002794
30,31,0.002694
40,41,0.002617


## Evaluate all 15 missing-feature combinations on the same target candidates
Ground-truth hidden values are removed before completion. They are used afterwards only for
reporting imputation quality, with no tuning. Known income labels supply the attack's
true-class features, as in the existing baseline. Each mask's scores and diagnostics are saved.


In [7]:
candidates = pd.concat([partitions['mia_members'].assign(membership=1),
    partitions['mia_nonmembers'].assign(membership=0)]).sample(frac=1,random_state=SEED)
assert candidates.membership.value_counts().to_dict()=={0:10000,1:10000}
assert candidates.index.tolist()==target_features.record_id.tolist()
results = run_mask_evaluations(completer,target,pp,candidates,attack,baseline,RUN_DIR,WALK_CONFIG)
display(results[['missing_count','missing_fields','accuracy','roc_auc','tpr_at_1pct_fpr_roc',
    'tpr_at_shadow_threshold','fpr_at_shadow_threshold']])


Missing-feature evaluation: COW


Attack feature extraction: 512/20,000


Attack feature extraction: 1,024/20,000


Attack feature extraction: 1,536/20,000


Attack feature extraction: 2,048/20,000


Attack feature extraction: 2,560/20,000


... (606 outputs left out)

Attack feature extraction: 19,456/20,000


Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


,missing_count,missing_fields,accuracy,roc_auc,tpr_at_1pct_fpr_roc,tpr_at_shadow_threshold,fpr_at_shadow_threshold
0,0,,0.60490,0.645108,0.0017,0.2181,0.1341
1,1,COW,0.57920,0.611342,0.0018,0.1826,0.1244
2,1,WKHP,0.56825,0.591970,0.0019,0.1752,0.1247
3,1,MAR,0.58185,0.609077,0.0019,0.1863,0.1262
4,1,SCHL,0.55530,0.573741,0.0016,0.1646,0.1239
5,2,"COW,WKHP",0.55410,0.567585,0.0026,0.1499,0.1218
6,2,"COW,MAR",0.56070,0.582700,0.0020,0.1616,0.1199
7,2,"COW,SCHL",0.54070,0.555991,0.0018,0.1439,0.1116
8,2,"WKHP,MAR",0.55610,0.573689,0.0023,0.1618,0.1218
9,2,"WKHP,SCHL",0.53610,0.540681,0.0018,0.1421,0.1198


## Save macro averages, configuration, and execution status

In [8]:
columns = ['accuracy','balanced_accuracy','roc_auc','tpr_at_1pct_fpr_roc',
    'tpr_at_shadow_threshold','fpr_at_shadow_threshold','imputation_encoded_mse']
averages = results.groupby('missing_count')[columns].mean()
averages.to_csv(RUN_DIR/'missing_count_averages.csv')
display(averages)
summary = dict(execution_status='all_cells_completed',target_year=2023,auxiliary_year=2014,
    target_variable='nominal PINCP > 50000',features=FEATURES,withheld_fields=WITHHELD,
    train_size=TRAIN_SIZE,n_shadow_models=16,shadow_algorithms=['random_forest','gradient_boosting','xgboost'],
    training=target_metadata,target_metrics=target_metrics,baseline_mia=baseline,
    split_config=SPLIT_CONFIG,walk_config=WALK_CONFIG,shadow_feature_records_per_membership=2000,
    ae=dict(epochs=50,learning_rate=.001,batch_size=256,encoder=[128,64],decoder=[64,128],
        loss='MSE on all encoded fields',auxiliary_training_records=len(completer.train_ids),
        auxiliary_validation_records=len(completer.validation_ids),
        masks='uniform 1..4 distinct original fields per row from all ten fields',
        decode='categorical argmax; numeric range clipping and integer rounding'),
    classifier_and_thresholds_frozen=True,mask_results=results.to_dict(orient='records'),
    averages_by_missing_count=averages.reset_index().to_dict(orient='records'))
(RUN_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Completed experiment saved to',RUN_DIR)


,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,tpr_at_shadow_threshold,fpr_at_shadow_threshold,imputation_encoded_mse
missing_count,,,,,,,
0,0.604900,0.604900,0.645108,0.001700,0.218100,0.134100,NaN
1,0.571150,0.571150,0.596532,0.001800,0.177175,0.124800,0.070026
2,0.548467,0.548467,0.561998,0.002117,0.150617,0.119200,0.078044
3,0.533288,0.533288,0.539711,0.002325,0.129725,0.115825,0.073548
4,0.521300,0.521300,0.523365,0.002800,0.114400,0.113400,0.070537


Completed experiment saved to artifacts\exp_3_missing_features_10inputs_2023_2014
